In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [3]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 5 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_5_data = {}

# 1. Load File Cimut
try:
    with open('fase_5_cimut.pkl', 'rb') as f:
        all_fase_5_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_5_afrida.pkl', 'rb') as f:
        all_fase_5_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_5_hanif.pkl'):
        with open('fase_5_hanif.pkl', 'rb') as f:
            all_fase_5_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 5 (SALING SILANG & AUTO-SKIP) 🚀 
⚠️ Gagal memuat file pkl Cimut: [Errno 2] No such file or directory: 'fase_5_cimut.pkl'
⚠️ Gagal memuat file pkl Afrida: [Errno 2] No such file or directory: 'fase_5_afrida.pkl'
✓ Berhasil memuat data hasil konversi Hanif.


In [4]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS FASE LOG & RAPOR GLOBAL (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Urutan di bawah ini disusun ketat lintas personel demi keselamatan relasi Foreign Key!
tables_to_insert_ordered = [
    # --- BLOK A: DATA MASTER KONFIGURASI FORMAT RAPOR INDUK (Karya Hanif) ---
    'rapor_format',             # Master template format rapor utama
    'rapor_format_sub',         # Sub-bab / kategori penilaian dalam format rapor
    'rapor_format_formula',     # Rumus / formula dasar kalkulasi nilai rapor
    'rapor_format_formula_sub', # Detail parameter sub-formula penilaian
    'rapor_level_config',       # Konfigurasi standar rapor berdasarkan tingkatan kelas
    'rapor_sub_level',          # Sub-tingkatan atau pengelompokan level rapor

    # --- BLOK B: DATA TRANSAKSIONAL RAPOR SISWA REAL (Karya Hanif) ---
    'rapor_siswa',              # Input data nilai rapor milik masing-masing siswa
    'rapor_siswa_file',         # Berkas / file PDF rapor siswa yang sudah di-generate
    'rapor_lacak',              # Log tracking / riwayat pembagian & perubahan rapor

    # --- BLOK C: AKADEMIK & OPERASIONAL SISWA (Karya Afrida) ---
    'presensi_siswa',           # Log kehadiran harian siswa di kelas
    'catatan_siswa',            # Catatan khusus / lembar BK untuk perkembangan siswa
    'followup_cs',              # Catatan tindak lanjut tim Customer Service ke wali siswa

    # --- BLOK D: LOG SISTEM, PASSPORT, & AUDIT TRAIL (Karya Cimut) ---
    'password_reset_tokens',    # Token keamanan reset kata sandi akun user
    'activity_log',             # Log jejak aktivitas sistem utama (modern)
    'log_aktivitas',            # Log jejak aktivitas sistem (legacy / versi lama)
    'jadwal_detail_logs'        # Catatan riwayat audit jika ada perubahan plot jadwal kelas
]

In [ ]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name]) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name])
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [6]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_5 = insert_data_with_preview_and_skip_v2(
    db_connection=db_new, 
    cursor=cursor_new, 
    tables_data=all_fase_5_data, 
    ordered_list=tables_to_insert_ordered
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)



 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ rapor_format: Sukses diproses! Sebanyak 45 baris sukses dimasukkan / di-skip aman.
  ✓ rapor_format_sub: Sukses diproses! Sebanyak 129 baris sukses dimasukkan / di-skip aman.
  ✓ rapor_format_formula: Sukses diproses! Sebanyak 3 baris sukses dimasukkan / di-skip aman.
  ✓ rapor_format_formula_sub: Sukses diproses! Sebanyak 1650 baris sukses dimasukkan / di-skip aman.
  ✓ rapor_level_config: Sukses diproses! Sebanyak 348 baris sukses dimasukkan / di-skip aman.
  ✓ rapor_siswa: Sukses diproses! Sebanyak 22837 baris sukses dimasukkan / di-skip aman.
  ✓ rapor_siswa_file: Sukses diproses! Sebanyak 1506 baris sukses dimasukkan / di-skip aman.
  ✓ rapor_lacak: Sukses diproses! Sebanyak 1366 baris sukses dimasukkan / di-skip aman.

🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):
  ℹ️  rapor_sub_level: DataFrame kosong (0 baris)
  ⚠️  presensi_siswa: Tidak ditemukan di file pkl
  ⚠️  catatan_sis

,id_rapor_format,id_kursus,judul_rapor
0,F00001,K00001,CLASSROOM ASSESSMENT
1,F00002,K00001,END OF TERM TEST
2,F00003,K00001,CLASS REMARKS
3,F00004,K00001,CLASSROOM ASSESSMENT
4,F00005,K00001,END OF TERM TEST


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: RAPOR_FORMAT_SUB]
--------------------------------------------------


,id_rapor_format_sub,id_rapor_format,sub_judul_rapor
0,D00001,F00001,Class Participation
1,D00002,F00001,Oral
2,D00003,F00001,Listening
3,D00005,F00002,Oral and Listening
4,D00006,F00004,Class Participation


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: RAPOR_SISWA_FILE]
--------------------------------------------------


,id_rapor_siswa_file,id_rapor_siswa,file_rapor_path
0,F00006,R005457,uploads/rapor/S0000329.jpeg
1,F00007,R005463,uploads/rapor/S0000474.jpeg
2,F00008,R005474,uploads/rapor/S0000481.jpeg
3,F00009,R005469,uploads/rapor/S0000475.jpeg
4,F00010,R005480,uploads/rapor/S0000482.jpeg


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: RAPOR_LACAK]
--------------------------------------------------


,id_rapor_lacak,id_siswa,id_jadwal,tanggal_terkirim,status_pengiriman,id_rapor_siswa_file
0,H00001,S0000609,J000000173,2024-11-18 10:57:59,Terkirim,F00658
1,H00005,S0000144,J000000352,2024-12-02 11:37:11,Terkirim,F00663
2,H00006,S0000714,J000000352,2024-12-02 11:37:12,Terkirim,F00664
3,H00007,S0000348,J000000352,2024-12-02 11:37:12,Terkirim,F00661
4,H00008,S0000556,J000000352,2024-12-02 11:37:13,Terkirim,F00662


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [7]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 5 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_5 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )